<a href="https://colab.research.google.com/github/evelynendindakimani/skillpath_recommender/blob/main/step1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load Dataset (Using a sample structure based on our proposal)
# You can replace this with your downloaded CSV file path later.
data = {
    'Course_ID': ['C101', 'C102', 'C103', 'C104', 'C105'],
    'Course_Name': [
        'Python for Everybody',
        'Machine Learning Specialization',
        'Deep Learning Specialization',
        'Web Development with Flask',
        'Data Structures and Algorithms in Python'
    ],
    'Difficulty': ['Beginner', 'Intermediate', 'Advanced', 'Beginner', 'Intermediate'],
    'Rating': [4.8, 4.9, 4.8, 4.6, 4.7],
    'Enrollment_Count': [15000, 12000, 8000, 9500, 11000],
    'Skills_Tags': [
        'Python Data Structures Programming',
        'Python Machine Learning Regression',
        'Neural Networks TensorFlow Python',
        'Python Flask Web Development HTML',
        'Python Algorithms Data Structures'
    ]
}

df = pd.DataFrame(data)
print("Dataset Loaded Successfully. Total Courses:", len(df))

# ---------------------------------------------------------
# 2. BASELINE RERECOMMENDER: Popularity-Based
# ---------------------------------------------------------
def recommend_popular(dataframe, top_n=3):
    """Recommends courses with the highest composite score of rating and enrollment count."""
    dataframe['Popularity_Score'] = (dataframe['Rating'] * 0.5) + (dataframe['Enrollment_Count'] / 10000 * 0.5)
    sorted_df = dataframe.sort_values(by='Popularity_Score', ascending=False)
    return sorted_df[['Course_ID', 'Course_Name', 'Difficulty', 'Rating']].head(top_n)

print("\n--- Popularity Baseline Recommendations ---")
print(recommend_popular(df, top_n=2))

# ---------------------------------------------------------
# 3. APPROACH 1: Content-Based Filtering (TF-IDF & Cosine Similarity)
# ---------------------------------------------------------
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['Skills_Tags'])

# Compute Cosine Similarity between courses
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

def recommend_content(course_name, dataframe=df, sim_matrix=cosine_sim, top_n=2):
    """Recommends similar courses based on textual skill tags matching."""
    if course_name not in dataframe['Course_Name'].values:
        return f"Course '{course_name}' not found in database."

    # Get index of the course
    idx = dataframe[dataframe['Course_Name'] == course_name].index[0]

    # Get similarity scores with other courses
    sim_scores = list(enumerate(sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get top_n similar courses (excluding the course itself)
    sim_scores = sim_scores[1:top_n+1]
    course_indices = [i[0] for i in sim_scores]

    return dataframe.iloc[course_indices][['Course_ID', 'Course_Name', 'Difficulty', 'Rating']]

print("\n--- Content-Based Recommendations (If you liked 'Python for Everybody') ---")
print(recommend_content('Python for Everybody', top_n=2))

Dataset Loaded Successfully. Total Courses: 5

--- Popularity Baseline Recommendations ---
  Course_ID                      Course_Name    Difficulty  Rating
0      C101             Python for Everybody      Beginner     4.8
1      C102  Machine Learning Specialization  Intermediate     4.9

--- Content-Based Recommendations (If you liked 'Python for Everybody') ---
  Course_ID                               Course_Name    Difficulty  Rating
4      C105  Data Structures and Algorithms in Python  Intermediate     4.7
1      C102           Machine Learning Specialization  Intermediate     4.9


In [3]:
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 35.3 MB/s eta 0:00:00


In [4]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate

# 1. Create a mock user-item interaction dataset (User IDs, Course IDs, and Ratings)
interaction_data = {
    'UserID': ['U1', 'U1', 'U2', 'U2', 'U3', 'U3', 'U4', 'U4', 'U5', 'U5'],
    'Course_ID': ['C101', 'C102', 'C101', 'C103', 'C102', 'C104', 'C101', 'C105', 'C103', 'C104'],
    'Rating': [5.0, 4.0, 3.0, 5.0, 4.5, 4.0, 2.0, 5.0, 4.0, 3.5]
}

df_interactions = pd.DataFrame(interaction_data)
print("Interaction Dataset Loaded. Total Ratings:", len(df_interactions))

# 2. Prepare data for the Surprise library
# Reader specifies the rating scale (from 1 to 5)
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_interactions[['UserID', 'Course_ID', 'Rating']], reader)

# 3. Train the Matrix Factorization Model using SVD (Singular Value Decomposition)
trainset = data.build_full_trainset()
model = SVD()
model.fit(trainset)

# 4. Predict a rating for a specific user and course
# Let's predict what rating User 'U1' would give to course 'C104' (which they haven't taken yet)
user_id = 'U1'
course_id = 'C104'
prediction = model.predict(user_id, course_id)

print(f"\n--- Matrix Factorization (SVD) Prediction ---")
print(f"Predicted rating for User {user_id} on Course {course_id}: {prediction.est:.2f}")

# 5. Evaluate model performance using cross-validation (RMSE and MAE)
print("\n--- Model Evaluation (Cross-Validation) ---")
cross_validate(model, data, measures=['RMSE', 'MAE'], cv=3, verbose=True)

Interaction Dataset Loaded. Total Ratings: 10

--- Matrix Factorization (SVD) Prediction ---
Predicted rating for User U1 on Course C104: 3.87

--- Model Evaluation (Cross-Validation) ---
Evaluating RMSE, MAE of algorithm SVD on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    1.0172  1.8858  0.9315  1.2782  0.4311  
MAE (testset)     0.8491  1.7677  0.7379  1.1182  0.4615  
Fit time          0.01    0.00    0.00    0.00    0.00    
Test time         0.00    0.00    0.00    0.00    0.00    


{'test_rmse': array([1.01720865, 1.88583925, 0.93147757]),
 'test_mae': array([0.84910855, 1.76774865, 0.73789155]),
 'fit_time': (0.005522727966308594,
  0.00023102760314941406,
  0.00017380714416503906),
 'test_time': (0.00010824203491210938,
  3.24249267578125e-05,
  2.7179718017578125e-05)}